In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!unzip -q "/content/drive/MyDrive/Colab Notebooks/datasets/AAT.zip" -d "/content/AAT"


In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


In [8]:
train_df = pd.read_csv("/content/AAT/dataset/train.csv")
test_df = pd.read_csv("/content/AAT/dataset/test.csv")

In [9]:
train_path = "/content/AAT/dataset/Train Images"
test_path = "/content/AAT/dataset/Test Images"

In [10]:
# 6. Prepare image data
train_df['Class'] = train_df['Class'].astype(str)

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_path,
    x_col='Image',
    y_col='Class',
    subset='training',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

val_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_path,
    x_col='Image',
    y_col='Class',
    subset='validation',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=test_path,
    x_col='Image',
    y_col=None,
    target_size=(128, 128),
    class_mode=None,
    shuffle=False
)

Found 4787 validated image filenames belonging to 4 classes.
Found 1196 validated image filenames belonging to 4 classes.
Found 3219 validated image filenames.


In [11]:
# 7. Build the CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(4, activation='softmax')  # 4 classes
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
# 8. Train the model
model.fit(train_gen, validation_data=val_gen, epochs=10)

Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


150/150 ━━━━━━━━━━━━━━━━━━━━ 12s 47ms/step - accuracy: 0.3916 - loss: 1.5964 - val_accuracy: 0.4707 - val_loss: 1.1971
Epoch 2/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.5006 - loss: 1.1546 - val_accuracy: 0.4607 - val_loss: 1.1946
Epoch 3/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.5781 - loss: 1.0137 - val_accuracy: 0.4925 - val_loss: 1.1515
Epoch 4/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.6570 - loss: 0.8593 - val_accuracy: 0.4833 - val_loss: 1.2123
Epoch 5/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.7680 - loss: 0.6226 - val_accuracy: 0.4958 - val_loss: 1.3344
Epoch 6/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.8331 - loss: 0.4549 - val_accuracy: 0.4916 - val_loss: 1.5189
Epoch 7/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.8832 - loss: 0.3338 - val_accuracy: 0.5042 - val_loss: 1.7597
Epoch 8/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.9182 - loss: 0.2398 - val_accuracy: 0.48

In [13]:
# 9. Predict on test data
preds = model.predict(test_gen)
pred_labels = np.argmax(preds, axis=1)

101/101 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step


In [14]:
# Map predictions to class names
label_map = {v: k for k, v in train_gen.class_indices.items()}
pred_classes = [label_map[i] for i in pred_labels]

In [15]:
# 10. Save to submission.csv
submission = pd.DataFrame({
    'Image': test_df['Image'],
    'Class': pred_classes
})
submission.to_csv("submission.csv", index=False)

In [16]:
# Check first few rows
submission.head()

,Image,Class
0,image6245.jpg,Food
1,image10409.jpg,Food
2,image8692.jpg,misc
3,image10517.jpg,Food
4,image2580.jpg,Decorationandsignage


In [17]:
# Check number of rows
print("Total rows in submission:", len(submission))

Total rows in submission: 3219


In [18]:
# Check for missing values
print("Missing values:\n", submission.isnull().sum())

Missing values:
 Image    0
Class    0
dtype: int64


In [19]:
# Optional: Compare with test.csv
test_df = pd.read_csv("/content/AAT/dataset/test.csv")
print("Submission matches test.csv:", submission['Image'].equals(test_df['Image']))


Submission matches test.csv: True


In [ ]:
from google.colab import files
files.download("submission.csv")
# Your submission.csv file should contain two columns:

# 📄 Format of submission.csv:
# Image	Class
# image0001.jpg	Food
# image0002.jpg	Attire
# image0003.jpg	Food
# image0004.jpg	misc
# image0005.jpg	Decorationandsignage
# ...	...

# ✅ Column Details:
# Image: The exact filename of each image from test.csv.
# Example: image1234.jpg

# Class: The predicted label for that image.
# One of:

# Food

# Attire

# Decorationandsignage

# misc


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
!ls


AAT  drive  sample_data  submission.csv


In [22]:
!jupyter nbconvert --to script "/content/drive/MyDrive/Colab Notebooks/Dl_AAT.ipynb"


[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/Dl_AAT.ipynb to script
[NbConvertApp] Writing 3329 bytes to /content/drive/MyDrive/Colab Notebooks/Dl_AAT.txt
